# Phase 26 — Uganda outcome: justification of the cutoff and sensitivity analysis
**Resolves:** R2-13 (cutoff >=4 "pre-registered" claim; clinical vs data-driven threshold). No pre-registration identifier exists, so the
manuscript must describe the cutoff as an **investigator-defined, data-driven (near-balanced) threshold** and drop the word "pre-registered".
This phase shows how conclusions change with the cutoff and with a continuous (non-dichotomised) analysis, on the SAME group-aware partitions.
 * RF with the tuned hyper-parameters (no retuning) for cutoffs >=3, >=4 (primary), >=5; threshold re-tuned on the calibration partition each time;
 * random-forest regression on the 0-9 score: Spearman rho and R^2 on the test set.
**Outputs:** `S2_uganda_score_distribution.csv`, `S2_uganda_cutoff_sensitivity.csv`, `S2_uganda_continuous_model.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from modeling import build_model_pipeline, tune_threshold, build_preprocessor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
d = DATASETS["uganda"]; S = load_split("uganda", ENV); feats = features_of("uganda")
assert "Anxiety" in S["train"].columns, "split files must contain the raw 'Anxiety' score (notebook 04 keeps it)"
params = load_rf_params("uganda", ENV)["rf"]
dist = pd.concat([S[k].assign(partition=k) for k in ["train", "cal", "test"]]).groupby(["partition", "Anxiety"]).size().unstack(0).fillna(0).astype(int)
dist["total"] = dist.sum(axis=1); dist["cum_pct_from_top"] = (dist["total"][::-1].cumsum()[::-1] / dist["total"].sum() * 100).round(1)
save_table(dist.reset_index(), ENV, "S2_uganda_score_distribution.csv"); print(dist)

rows = []
for cut in [3, 4, 5]:
    ytr, yca, yte = [(S[k]["Anxiety"] >= cut).astype(int) for k in ["train", "cal", "test"]]
    pipe = build_model_pipeline("rf", params, d["numeric"], d["categorical"], SEED); pipe.fit(S["train"][feats], ytr)
    thr, _ = tune_threshold(yca, pipe.predict_proba(S["cal"][feats])[:, 1])
    p = pipe.predict_proba(S["test"][feats])[:, 1]
    est, lo, hi = bootstrap_ci_full(yte.values, p, B=B, seed=SEED)
    m = operating_metrics(yte.values, p, thr)
    rows.append(dict(cutoff=f">= {cut}", primary=(cut == 4), prevalence_test=yte.mean(), threshold=round(float(thr), 2), auc=est, ci_lower=lo, ci_upper=hi,
                     balanced_accuracy=m["balanced_accuracy"], sensitivity=m["sensitivity"], specificity=m["specificity"]))
sens = pd.DataFrame(rows); save_table(sens, ENV, "S2_uganda_cutoff_sensitivity.csv"); print(sens.round(3))

pre = build_preprocessor(d["numeric"], d["categorical"], scale_numeric=False)
reg = Pipeline([("pre", pre), ("rf", RandomForestRegressor(n_estimators=min(int(params.get("n_estimators", 300)), 300), max_depth=params.get("max_depth"),
                                                          min_samples_leaf=params.get("min_samples_leaf", 1), random_state=SEED, n_jobs=-1))])
reg.fit(S["train"][feats], S["train"]["Anxiety"]); pr = reg.predict(S["test"][feats])
rho, p_rho = spearmanr(pr, S["test"]["Anxiety"])
cont = pd.DataFrame([dict(model="RF regression on 0-9 anxiety score", spearman_rho=rho, spearman_p=p_rho, r2_test=r2_score(S["test"]["Anxiety"], pr), n_test=len(pr))])
save_table(cont, ENV, "S2_uganda_continuous_model.csv"); cont

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_uganda_score_distribution.csv  (10 rows)
partition  cal  test  train  total  cum_pct_from_top
Anxiety                                             
0           73   101    299    473             100.0
1          141   119    394    654              96.7
2          440   402   1128   1970              92.1
3          819   853   2534   4206              78.4
4          556   547   1667   2770              49.0
5          454   415   1327   2196              29.7
6          307   315    952   1574              14.4
7           70    74    259    403               3.4
8           12    12     43     67               0.6
9            0     3      9     12               0.1
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_uganda_cutoff_sensitivity.csv  (3 rows)
  cutoff  primary  prevalence_test  threshold    auc  ci_lower  ci_upper  \
0   >= 3    False            0.781       0.52  0.675     0.652     0.697   
1

,model,spearman_rho,spearman_p,r2_test,n_test
0,RF regression on 0-9 anxiety score,0.301425,9.398796e-61,0.093384,2841


**Wording guide:** "The Uganda outcome was a binary indicator of an anxiety severity score >= 4, an investigator-defined, data-driven cutoff
yielding approximately balanced classes (48-49% positive); it is not a clinically validated threshold, and no pre-registration record is available.
Sensitivity analyses at >= 3 and >= 5 and a continuous-score model are reported in Table S-x." Report the AUC range across cutoffs honestly.